# 01 · Data Cleaning

**Input:** `data/raw/superstore_raw.csv` (10,800 rows as received)
**Output:** `data/cleaned/superstore_clean.csv`

Profile the extract, decide what is wrong with it, and fix it — documenting every
decision. The reusable implementation is `src/clean_data.py`; this notebook shows
the reasoning behind it.

In [1]:
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd
import seaborn as sns

warnings.filterwarnings("ignore")
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

BLUE, TEAL, AMBER, PURPLE, RED = "#2f6fed", "#16a888", "#f5b820", "#7b45c9", "#e03131"
sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 140, "savefig.bbox": "tight",
    "font.size": 10, "axes.titlesize": 12, "axes.titleweight": "bold",
    "axes.edgecolor": "#d7dbe3", "grid.color": "#eef0f4", "axes.facecolor": "white",
})
IMAGES = ROOT / "images"; IMAGES.mkdir(exist_ok=True)
pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
money = mticker.FuncFormatter(lambda v, _: f"${v/1000:,.0f}K")

## 1. Load and inspect

In [2]:
# Superstore ships as latin-1. utf-8 fails on customer names with accented characters.
raw = pd.read_csv(ROOT / "data" / "raw" / "superstore_raw.csv", encoding="latin-1")
print(f"Shape: {raw.shape[0]:,} rows x {raw.shape[1]} columns")
raw.head(3)

Shape: 10,800 rows x 21 columns


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
0,1,CA-2017-152156,11/8/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,"42,420.00",South,FUR-BO-10001798,Furniture,Bookcases,Bush Somerset Collection Bookcase,261.96,2.00,0.00,41.91
1,2,CA-2017-152156,11/8/2017,11/11/2017,Second Class,CG-12520,Claire Gute,Consumer,United States,Henderson,...,"42,420.00",South,FUR-CH-10000454,Furniture,Chairs,"Hon Deluxe Fabric Upholstered Stacking Chairs,...",731.94,3.00,0.00,219.58
2,3,CA-2017-138688,6/12/2017,6/16/2017,Second Class,DV-13045,Darrin Van Huff,Corporate,United States,Los Angeles,...,"90,036.00",West,OFF-LA-10000240,Office Supplies,Labels,Self-Adhesive Address Labels for Typewriters b...,14.62,2.00,0.00,6.87


## 2. Data profiling

In [3]:
profile = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "nulls": raw.isna().sum(),
    "null_%": (raw.isna().mean() * 100).round(2),
    "unique": raw.nunique(),
    "sample": [raw[c].dropna().iloc[0] if raw[c].notna().any() else "-" for c in raw.columns],
})
profile

,dtype,nulls,null_%,unique,sample
Row ID,str,0,0.00,10001,1
Order ID,str,0,0.00,5015,CA-2017-152156
Order Date,str,806,7.46,1236,11/8/2017
Ship Date,str,806,7.46,1334,11/11/2017
Ship Mode,str,806,7.46,4,Second Class
Customer ID,str,806,7.46,793,CG-12520
Customer Name,str,806,7.46,793,Claire Gute
Segment,str,806,7.46,3,Consumer
Country,str,806,7.46,1,United States
City,str,806,7.46,531,Henderson


### Reading the null pattern

19 columns each report exactly **806** nulls, and `Postal Code` reports 817. Identical
counts across unrelated columns is not scattered data entry error — it is whole records
arriving empty. Confirm before deciding what to do.

In [4]:
shells = raw[raw["Sales"].isna()]
print(f"Rows with no Sales value : {len(shells):,}")
print(f"Of those, Order ID present: {shells['Order ID'].notna().sum():,}")
print(f"Columns null in ALL shells: "
      f"{(shells.isna().all() | (shells.notna().sum() == 0)).sum()}")
shells.head(3)

Rows with no Sales value : 806
Of those, Order ID present: 806
Columns null in ALL shells: 19


,Row ID,Order ID,Order Date,Ship Date,Ship Mode,Customer ID,Customer Name,Segment,Country,City,...,Postal Code,Region,Product ID,Category,Sub-Category,Product Name,Sales,Quantity,Discount,Profit
9994,Person,Region,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9995,Anna Andreadi,West,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9996,Chuck Magee,East,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**Confirmed.** These rows carry an Order ID and nothing else. They are not missing values
to impute — there is no transaction to reconstruct. They are dropped.

The 11 extra `Postal Code` nulls are a separate issue, examined in step 5.

In [5]:
print(f"Exact duplicate rows: {raw.duplicated().sum():,}")
print(f"All duplicates inside the shell rows: "
      f"{raw[raw.duplicated()]['Sales'].isna().all()}")

Exact duplicate rows: 504
All duplicates inside the shell rows: True


## 3. Cleaning

In [6]:
df = raw.copy()
df.columns = (df.columns.str.strip().str.lower()
                .str.replace(r"[ \-]", "_", regex=True))

before = len(df)
critical = ["order_date", "customer_id", "product_id", "sales", "quantity"]

df = df.dropna(subset=critical, how="all")
print(f"After dropping shell rows : {len(df):,}  (-{before - len(df):,})")

n = len(df)
df = df.drop_duplicates(subset=[c for c in df.columns if c != "row_id"])
print(f"After dropping duplicates : {len(df):,}  (-{n - len(df):,})")

After dropping shell rows : 9,994  (-806)
After dropping duplicates : 9,993  (-1)


## 4. Type correction

In [7]:
# Dates arrive in mixed formats, so let pandas infer per value rather than forcing one.
for col in ["order_date", "ship_date"]:
    df[col] = pd.to_datetime(df[col], format="mixed", errors="coerce")

for col in ["sales", "profit", "discount"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df["quantity"] = pd.to_numeric(df["quantity"], errors="coerce").astype("Int64")

print(f"Unparsed dates: {df.order_date.isna().sum()}")
print(f"Date range    : {df.order_date.min():%Y-%m-%d} to {df.order_date.max():%Y-%m-%d}")
df[["order_date", "ship_date", "sales", "quantity", "discount", "profit"]].dtypes

Unparsed dates: 0
Date range    : 2015-01-03 to 2018-12-30


order_date    datetime64[us]
ship_date     datetime64[us]
sales                float64
quantity               Int64
discount             float64
profit               float64
dtype: object

### Postal codes are identifiers, not numbers

Read as integers, `01810` becomes `1810`. Cast to text and zero-pad.

In [8]:
df["postal_code"] = (df["postal_code"].astype("Float64").astype("Int64")
                     .astype(str).str.zfill(5).replace("<NA>", None))

# Trim stray whitespace in every text column.
text_cols = [c for c in df.columns
             if pd.api.types.is_string_dtype(df[c]) or df[c].dtype == "object"]
for col in text_cols:
    df[col] = df[col].astype("string").str.strip()

print(f"Still missing a postal code: {df.postal_code.isna().sum()}")
df.loc[df.postal_code.isna(), ["city", "state"]].drop_duplicates()

Still missing a postal code: 11


,city,state
2234,Burlington,Vermont


## 5. The Burlington problem

All 11 remaining gaps are **Burlington, Vermont**. Vermont ZIP codes begin with a zero
(Burlington is 05401), so the leading zero was almost certainly stripped by whatever
system exported this file, and the value was lost rather than corrupted.

Two options:

1. **Impute from the data** — take the most common postal code for that city and state.
   Reusable and defensible, but here every Burlington VT row is missing, so there is
   nothing to impute from.
2. **Fill from an external source** — we happen to know it is 05401.

The pipeline does (1) and leaves the residual null. Option (2) would put a value into a
file labelled "cleaned" that cannot be traced to the source data, and `postal_code` is
used in no analysis — `state` is the geographic grain. The gap is documented instead.

In [9]:
missing_before = df.postal_code.isna().sum()
mode_by_city = (df.dropna(subset=["postal_code"])
                  .groupby(["state", "city"])["postal_code"]
                  .agg(lambda s: s.mode().iat[0]))
keys = pd.MultiIndex.from_arrays([df["state"], df["city"]])
df["postal_code"] = df["postal_code"].fillna(pd.Series(keys.map(mode_by_city), index=df.index))

print(f"Imputed from city/state: {missing_before - df.postal_code.isna().sum()}")
print(f"Documented as unknown  : {df.postal_code.isna().sum()}")

Imputed from city/state: 0
Documented as unknown  : 11


## 6. Validation

In [10]:
checks = [
    ("No null order IDs",              df.order_id.isna().sum()),
    ("No null sales",                  df.sales.isna().sum()),
    ("No duplicates",                  df.drop(columns=["row_id"]).duplicated().sum()),
    ("Sales non-negative",             (df.sales < 0).sum()),
    ("Quantity positive",              (df.quantity <= 0).sum()),
    ("Discount within 0-1",            (~df.discount.between(0, 1)).sum()),
    ("Ship date >= order date",        (df.ship_date < df.order_date).sum()),
    ("Country single-valued",          (df.country != "United States").sum()),
    ("Region in expected set",         (~df.region.isin(["West","East","Central","South"])).sum()),
]
results = pd.DataFrame(checks, columns=["rule", "violations"])
results["status"] = np.where(results.violations == 0, "PASS", "FAIL")
results

,rule,violations,status
0,No null order IDs,0,PASS
1,No null sales,0,PASS
2,No duplicates,0,PASS
3,Sales non-negative,0,PASS
4,Quantity positive,0,PASS
5,Discount within 0-1,0,PASS
6,Ship date >= order date,0,PASS
7,Country single-valued,0,PASS
8,Region in expected set,0,PASS


## 7. Result

In [11]:
summary = pd.Series({
    "Rows in":        f"{len(raw):,}",
    "Rows out":       f"{len(df):,}",
    "Rows dropped":   f"{len(raw) - len(df):,}",
    "Total sales":    f"${df.sales.sum():,.2f}",
    "Total profit":   f"${df.profit.sum():,.2f}",
    "Profit margin":  f"{df.profit.sum()/df.sales.sum()*100:.2f}%",
    "Orders":         f"{df.order_id.nunique():,}",
    "Customers":      f"{df.customer_id.nunique():,}",
}, name="Value").to_frame()
summary

,Value
Rows in,"10,800"
Rows out,"9,993"
Rows dropped,807
Total sales,"$2,296,919.49"
Total profit,"$286,409.08"
Profit margin,12.47%
Orders,"5,009"
Customers,793


**9,993 clean transactions**, matching the canonical Superstore row count.

Revenue totals are unchanged by the cleaning, because every dropped row had no sales value.
That is the check worth stating: cleaning removed noise from the row counts without
altering a single financial figure.

→ Continue to `02_feature_engineering.ipynb`